# Modelagem Preditiva do Valor de Aluguéis

Este notebook tem como objetivo desenvolver e avaliar modelos de Machine Learning para prever o valor de aluguel (`rent`) de imóveis na cidade de São Paulo.

A etapa de modelagem será realizada a partir do dataset resultante do processo de Data Preparation (`data_cleaned`).

## Objetivo

O objetivo desta etapa é construir modelos de regressão capazes de estimar o valor do aluguel de um imóvel a partir de suas características estruturais e do seu tipo.

A variável `rent` será utilizada como variável target, enquanto as demais variáveis selecionadas serão utilizadas como features preditoras. A escolha de `rent` como variável target está alinhada ao objetivo principal do projeto: prever o valor do aluguel de imóveis a partir de suas características disponíveis no momento da predição.

Durante as etapas anteriores do projeto, foi identificada a variável `total`, que representa o valor total associado ao imóvel. Dessa forma, a utilização de `total` como variável preditora forneceria ao modelo uma informação diretamente relacionada ao próprio target que se deseja prever, situação que caracteriza data leakage. Além disso, as variáveis `address` e `district` não serão utilizadas por apresentarem alta cardinalidade e pela insuficiência de dados para transformar estas variáveis categóricas em informações úteis para o modelo.

Inicialmente, será desenvolvido um modelo baseline utilizando as seguintes características:

- `area`: área do imóvel em metros quadrados;
- `bedrooms`: número de quartos;
- `garage`: número de vagas de garagem;
- `type`: tipo do imóvel.

A escolha inicial dessas variáveis busca estabelecer uma base simples e interpretável para a modelagem. Posteriormente, novas features poderão ser incorporadas e seus impactos sobre o desempenho preditivo poderão ser avaliados individualmente.

## Tratamento da variável categórica `type`

A variável `type` será mantida como feature do modelo. Durante a EDA, observou-se que os diferentes tipos de imóveis apresentam distribuições distintas de características estruturais e valores de aluguel. Portanto, essa variável pode fornecer informação relevante para a previsão.

As categorias presentes no dataset representam diferentes tipos de imóveis, como:

- Apartamento;
- Casa;
- Casa em condomínio;
- Studio e kitnet.

Como modelos de Machine Learning geralmente não interpretam diretamente categorias textuais, a variável `type` será posteriormente transformada utilizando **One-Hot Encoding**.

In [1]:
# Importação de bibliotecas
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.dummy import DummyRegressor

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

In [2]:
# Bibliotecas de processamento
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
# Carregar dados limpos
df = pd.read_csv("../data/processed/data_cleaned.csv")

# Separar features que não entrarão no modelo
X = df.drop(
    columns=[
        "rent",
        "total",
        "address",
        "district"
    ]
)

# Target
y = df["rent"]

In [4]:
X.head()

,area,bedrooms,garage,type
0,21,1,0,Studio e kitnet
1,15,1,1,Studio e kitnet
2,18,1,0,Apartamento
3,56,2,2,Casa em condomínio
4,19,1,0,Studio e kitnet


## Modelagem sem novas features

In [5]:
# Dividir treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

Inicialmente foi realizado uma divisão do dataset apenas uma vez baseado no `train_test_split` para avaliar os modelos bases e entender qual se adequa mais aos dados. Entretanto, será implementado futuramente um **K-Fold Cross Validation**, garantindo que não há viés em divisões específicas.

### Pré-processamento das colunas do dataset com ColumnTransformer

A ideia é padronizar os dados numéricos com `StandardScaler()` de maneira que uma variável não "pese" mais que a outra no modelo. Adicionalmente, também é realizado um processo de One Hot Enconding na variável categórica `type`.

In [6]:
numeric_features = [
    "area",
    "bedrooms",
    "garage"
]

categorical_features = [
    "type"
]

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [8]:
X_train_processed = preprocessor.transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

In [9]:
feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(feature_names)

['num__area' 'num__bedrooms' 'num__garage' 'cat__type_Apartamento'
 'cat__type_Casa' 'cat__type_Casa em condomínio'
 'cat__type_Studio e kitnet']


In [10]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_train_processed_df.head()

,num__area,num__bedrooms,num__garage,cat__type_Apartamento,cat__type_Casa,cat__type_Casa em condomínio,cat__type_Studio e kitnet
6385,-0.526285,0.035516,-0.059238,1.0,0.0,0.0,0.0
9894,1.871716,2.184993,0.815671,0.0,1.0,0.0,0.0
3031,-0.241776,1.110254,-0.059238,1.0,0.0,0.0,0.0
5566,-0.038556,1.110254,-0.059238,1.0,0.0,0.0,0.0
6326,0.029184,0.035516,-0.934146,1.0,0.0,0.0,0.0


In [11]:
X_train.head()

,area,bedrooms,garage,type
6385,46,2,1,Apartamento
9894,223,4,2,Casa
3031,67,3,1,Apartamento
5566,82,3,1,Apartamento
6326,87,2,0,Apartamento


### Modelo 01: DummyRegressor (Baseline)

#### Treinamento do Modelo

In [12]:
baseline = DummyRegressor(strategy="mean")

baseline.fit(
    X_train_processed,
    y_train
)

y_pred_baseline = baseline.predict(
    X_test_processed
)

#### Métricas de Avaliação

In [13]:
mae_baseline = mean_absolute_error(
    y_test,
    y_pred_baseline
)

rmse_baseline = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_baseline
    )
)

r2_baseline = r2_score(
    y_test,
    y_pred_baseline
)

In [14]:
print(
    "MAE Baseline:",
    mae_baseline
)

print(
    "RMSE Baseline:",
    rmse_baseline
)

print(
    "R² Baseline:",
    r2_baseline
)

MAE Baseline: 1826.223965613672
RMSE Baseline: 2619.8292210387426
R² Baseline: -1.7038697572413497e-05


### Modelo 02: Linear Regression

### Treinamento do Modelo

In [15]:
linear_model = LinearRegression()

linear_model.fit(
    X_train_processed,
    y_train
)

y_pred_linear = linear_model.predict(
    X_test_processed
)

#### Métricas de Avaliação

In [16]:
mae_linear = mean_absolute_error(
    y_test,
    y_pred_linear
)

rmse_linear = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_linear
    )
)

r2_linear = r2_score(
    y_test,
    y_pred_linear
)

In [17]:
print(
    "MAE Linear Regression:",
    mae_linear
)

print(
    "RMSE Linear Regression:",
    rmse_linear
)

print(
    "R² Linear Regression:",
    r2_linear
)

MAE Linear Regression: 1207.4100313315687
RMSE Linear Regression: 1760.9429467407692
R² Linear Regression: 0.5481939867873397


### Modelo 03: Random Forest

#### Treinamento do Modelo

In [18]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_processed,
    y_train
)

y_pred_rf = rf_model.predict(
    X_test_processed
)

#### Métricas de Avaliação

In [19]:
mae_rf = mean_absolute_error(
    y_test,
    y_pred_rf
)

rmse_rf = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_rf
    )
)

r2_rf = r2_score(
    y_test,
    y_pred_rf
)

In [20]:
print(
    "MAE Random Forest:",
    mae_rf
)

print(
    "RMSE Random Forest:",
    rmse_rf
)

print(
    "R² Random Forest:",
    r2_rf
)

MAE Random Forest: 1124.0191807590127
RMSE Random Forest: 1753.413307318325
R² Random Forest: 0.5520494929320796


### Modelo 04: Gradient Boosting

#### Treinamento do Modelo

In [21]:
gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

In [22]:
gb_model.fit(
    X_train_processed,
    y_train
)

,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, in

In [23]:
y_pred_gb = gb_model.predict(
    X_test_processed
)

#### Métricas de Avaliação

In [24]:
mae_gb = mean_absolute_error(
    y_test,
    y_pred_gb
)

rmse_gb = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_gb
    )
)

r2_gb = r2_score(
    y_test,
    y_pred_gb
)

In [25]:
print(
    "MAE Gradient Boosting:",
    mae_gb
)

print(
    "RMSE Gradient Boosting:",
    rmse_gb
)

print(
    "R² Gradient Boosting:",
    r2_gb
)

MAE Gradient Boosting: 1092.549936496278
RMSE Gradient Boosting: 1655.6186137907164
R² Gradient Boosting: 0.6006239319746026


### Modelo 05: XGBoost

#### Treinamento do Modelo

In [26]:
xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(
    X_train_processed,
    y_train
)

y_pred_xgb = xgb_model.predict(
    X_test_processed
)

#### Métricas de Avaliação

In [27]:
mae_xgb = mean_absolute_error(
    y_test,
    y_pred_xgb
)

rmse_xgb = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_xgb
    )
)

r2_xgb = r2_score(
    y_test,
    y_pred_xgb
)

In [28]:
print(
    "MAE XGBoost:",
    mae_xgb
)

print(
    "RMSE XGBoost:",
    rmse_xgb
)

print(
    "R² XGBoost:",
    r2_xgb
)

MAE XGBoost: 1089.44873046875
RMSE XGBoost: 1649.4877992880092
R² XGBoost: 0.6035762429237366


## Resultados dos Modelos

In [29]:
results = pd.DataFrame({
    "Model": [
        "DummyRegressor",
        "LinearRegression",
        "RandomForest",
        "GradientBoosting",
        "XGBoost"
    ],
    "MAE": [
        mae_baseline,
        mae_linear,
        mae_rf,
        mae_gb,
        mae_xgb
    ],
    "RMSE": [
        rmse_baseline,
        rmse_linear,
        rmse_rf,
        rmse_gb,
        rmse_xgb
    ],
    "R2": [
        r2_baseline,
        r2_linear,
        r2_rf,
        r2_gb,
        r2_xgb
    ]
})

In [30]:
results.sort_values(
    "RMSE"
)

,Model,MAE,RMSE,R2
4,XGBoost,1089.448730,1649.487799,0.603576
3,GradientBoosting,1092.549936,1655.618614,0.600624
2,RandomForest,1124.019181,1753.413307,0.552049
1,LinearRegression,1207.410031,1760.942947,0.548194
0,DummyRegressor,1826.223966,2619.829221,-0.000017


In [31]:
evaluation_df = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred_xgb
})

In [32]:
evaluation_df["error"] = (
    evaluation_df["actual"]
    - evaluation_df["predicted"]
)

In [33]:
evaluation_df["absolute_error"] = (
    evaluation_df["error"]
    .abs()
)

In [34]:
evaluation_df.sort_values(
    "absolute_error",
    ascending=False
).head(20)

,actual,predicted,error,absolute_error
8634,15000,3625.956299,11374.043701,11374.043701
3131,15000,4606.423828,10393.576172,10393.576172
6835,15000,6044.799316,8955.200684,8955.200684
7224,15000,6079.859863,8920.140137,8920.140137
9848,15000,6339.281738,8660.718262,8660.718262
10318,15000,6475.806152,8524.193848,8524.193848
10317,12000,4108.673340,7891.326660,7891.326660
758,12000,4238.730957,7761.269043,7761.269043
9770,14500,6767.166504,7732.833496,7732.833496
7602,11000,3516.307861,7483.692139,7483.692139


## Análise inicial dos modelos de regressão

Após a etapa de preparação dos dados e aplicação do pré-processamento, foram avaliados cinco modelos de regressão:

- `DummyRegressor`, utilizado como baseline;
- `LinearRegression`, como modelo linear de referência;
- `RandomForestRegressor`, baseado em um conjunto de árvores de decisão;
- `GradientBoostingRegressor`, baseado em boosting sequencial;
- `XGBRegressor`, implementação otimizada de Gradient Boosting.

Nesta primeira etapa, foram utilizadas apenas as características originais selecionadas para a modelagem:

- `area`;
- `bedrooms`;
- `garage`;
- `type`.

O objetivo desta primeira rodada foi estabelecer um **baseline de desempenho**.

### Resultados

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| XGBoost | 1089.448730 | 1649.487799 | 0.603576 |
| GradientBoosting | 1092.549936 | 1655.618614 | 0.600624 |
| RandomForest | 1124.019181 | 1753.413307 | 0.552049 |
| LinearRegression | 1207.410031 | 1760.942947 | 0.548194 |
| DummyRegressor | 1826.223966 | 2619.829221 | -0.000017 |

Os resultados indicam que os modelos de Machine Learning apresentaram desempenho superior ao `DummyRegressor`, utilizado como baseline. A `LinearRegression` apresentou desempenho inferior aos modelos baseados em árvores, sugerindo que a relação entre as características dos imóveis e o valor do aluguel apresenta componentes não lineares que não são completamente representados por um modelo linear.

Entre os modelos avaliados, os métodos baseados em boosting apresentaram os melhores resultados. O `GradientBoostingRegressor` obteve MAE de aproximadamente R$ 1.093, RMSE de aproximadamente R$ 1.656 e R² de aproximadamente 0,601. O melhor resultado foi obtido pelo `XGBRegressor`, com MAE de aproximadamente R$ 1.089, RMSE de aproximadamente R$ 1.649 e R² de aproximadamente 0,604.

Embora o XGBoost tenha apresentado o melhor desempenho inicial, sua vantagem em relação ao Gradient Boosting foi pequena: aproximadamente R$ 3,10 de diferença no MAE e R$ 6,13 no RMSE. Portanto, nesta etapa preliminar, não é possível afirmar que o XGBoost seja significativamente superior ao Gradient Boosting.

Para os próximos experimentos, o XGBoost será adotado como modelo principal, permitindo investigar de forma controlada o impacto das novas features desenvolvidas durante a etapa de Feature Engineering.

---

## Análise dos erros de previsão

Após a avaliação inicial, foram analisados os maiores erros absolutos produzidos pelo XGBoost, comparando os valores reais de aluguel (`actual`) com os valores previstos (`predicted`).

O erro de previsão foi definido como $error = actual - predicted$, e o erro absoluto como $absolute\_error = |actual - predicted|$

A análise dos maiores erros revelou um padrão relevante: a maior parte dos erros absolutos mais elevados está associada a imóveis com valores de aluguel entre aproximadamente R$ 10.000 e R$ 15.000. Nesses casos, o modelo tende a subestimar significativamente o valor real do aluguel.

O maior erro observado correspondeu a um imóvel com aluguel real de R$ 15.000, para o qual o modelo previu aproximadamente R$ 3.626, resultando em um erro absoluto superior a R$ 11.000. Outros imóveis com aluguéis de R$ 12.000, R$ 14.000 e R$ 15.000 também apresentaram subestimações significativas. Esse comportamento sugere que o modelo possui dificuldades para representar adequadamente a região superior da distribuição de aluguéis. Entretanto, esses registros não devem ser considerados automaticamente como erros ou outliers inválidos. Um imóvel com aluguel elevado pode representar corretamente um imóvel de alto padrão.

Portanto, é necessário distinguir entre:

- **outlier estatístico**, um valor extremo em relação à distribuição;
- **erro de registro**, um dado incorreto ou inconsistente;
- **observação válida de alto padrão**, que representa um segmento real do mercado imobiliário.

A remoção indiscriminada de imóveis com aluguéis elevados poderia melhorar artificialmente as métricas de avaliação, mas também reduzir a capacidade do modelo de generalizar para imóveis de alto padrão.

A análise dos erros, portanto, levanta a seguinte hipótese:

> Os maiores erros do modelo podem estar associados à dificuldade de identificar e caracterizar imóveis de alto padrão, e não necessariamente à presença de registros inválidos.

Também foi identificado um caso de erro na direção oposta, em que um imóvel com aluguel real de R$ 4.500 foi previsto em aproximadamente R$ 11.061. Esse caso demonstra que o modelo também pode superestimar determinados imóveis, embora os maiores erros estejam predominantemente associados à subestimação de imóveis com aluguéis elevados.

---

## Próximos experimentos

Os resultados obtidos nesta primeira rodada indicam que os modelos baseados em árvores e boosting conseguem representar melhor as relações não lineares presentes nos dados. O XGBoost apresentou o melhor desempenho inicial e será utilizado como modelo principal nos experimentos seguintes.

O objetivo será investigar se as features desenvolvidas durante a etapa de Feature Engineering fornecem informações adicionais capazes de melhorar a identificação do porte e do padrão relativo dos imóveis.

A sequência planejada é:

### Experimento 1 — Baseline

Utilizar as características originais:

- `area`;
- `bedrooms`;
- `garage`;
- `type`.

Os resultados desta etapa constituem o baseline inicial do XGBoost:

- MAE ≈ R$ 1.089;
- RMSE ≈ R$ 1.649;
- R² ≈ 0,604.

### Experimento 2 — Standard Score global

Adicionar:

- `standard_score`;
- `high_standard`.

O objetivo é verificar se uma medida global de porte do imóvel e uma classificação empírica de alto padrão contribuem para melhorar a capacidade preditiva do modelo.

### Experimento 3 — Standard Score por tipo

Adicionar:

- `standard_score_type`;
- `high_standard_type`.

O objetivo é avaliar se a caracterização do porte relativo dentro de cada categoria de imóvel fornece informações adicionais ao modelo.

### Validação dos experimentos

Os experimentos serão inicialmente comparados utilizando as métricas:

- **MAE**, para avaliar o erro absoluto médio em reais;
- **RMSE**, para avaliar a influência de erros de maior magnitude;
- **R²**, para avaliar a proporção da variabilidade do aluguel explicada pelo modelo.

Além da avaliação por uma única divisão entre treino e teste, será realizada **validação cruzada K-Fold** no conjunto de treinamento. Essa etapa permitirá avaliar a estabilidade do XGBoost em diferentes subconjuntos dos dados e obter uma estimativa mais robusta de seu desempenho médio. A validação cruzada será utilizada nos experimentos de melhoria para comparar de forma consistente o baseline com as diferentes configurações de Feature Engineering. O conjunto de teste será mantido separado para a avaliação final do modelo selecionado.

Os experimentos de melhoria e a validação cruzada serão desenvolvidos no notebook `06_model_improvement.ipynb`.